# Pharmacy Demand Forecasting
## Forecasting Daily Pharmaceutical Product Sales

**Notebook #15 of 50 — Time Series Forecasting Portfolio**

| | |
|---|---|
| **Dataset** | Pharma Sales Data (`milanzdravkovic/pharma-sales-data`) |
| **Target** | Daily sales of drug category `M01AB` (anti-inflammatory, non-steroidal) |
| **Date column** | `datum` |
| **Frequency** | Daily (`D`) |
| **Primary TS Library** | StatsForecast (AutoARIMA + AutoETS + AutoTheta) |

## Learning Objectives
1. Work with real pharmaceutical sales data (ATC drug category codes)
2. Understand multi-series pharmaceutical demand patterns
3. Forecast a single high-volume drug category as the primary task
4. Explore the challenge of seasonality in pharmaceutical demand (flu season, allergy season)
5. Discuss implications of over- and under-stocking in a pharmacy context

## Why This Project Matters
Pharmacy stock-outs have real clinical consequences — a patient who cannot get their prescribed
medication faces delayed treatment, pain, or worse. On the other side, over-ordering leads to
expired stock and wasted budget. Accurate demand forecasting enables:
- **Just-in-time ordering** from distributors, reducing carrying costs
- **Patient safety** through guaranteed availability of critical medications
- **Seasonal preparedness** (e.g. stocking up on antihistamines before allergy season)

This project demonstrates how time series forecasting applies to pharmaceutical supply chain management.

## Problem Statement
A pharmacy needs to forecast daily demand for key drug categories to:
- **Maintain adequate stock levels** (avoid stock-outs)
- **Optimise order quantities** from the distributor
- **Reduce waste** from expired stock

> Goal: Forecast the next 14 days of daily sales for `M01AB` (NSAIDs like ibuprofen/diclofenac).

## Dataset Overview
**Source:** Kaggle — `milanzdravkovic/pharma-sales-data`

**License:** CC BY 4.0

### ATC Drug Category Reference
| Code | Drug Type | Seasonal Pattern |
|------|----------|-----------------|
| `M01AB` | NSAIDs (anti-inflammatory) — diclofenac, indomethacin | Year-round, slight winter peak |
| `M01AE` | Propionic acid derivatives — ibuprofen, naproxen | Year-round |
| `N02BA` | Salicylic acid — aspirin | Year-round |
| `N02BE` | Anilides — paracetamol | Winter peak (flu/cold season) |
| `N05B` | Anxiolytics (benzodiazepines) | Relatively flat |
| `N05C` | Hypnotics/sedatives | Relatively flat |
| `R03` | Bronchodilators / asthma | Winter + spring (pollen) peak |
| `R06` | Antihistamines | Spring–summer peak (allergy season) |

### Date column
`datum` — format `YYYY-MM-DD HH:MM:SS`, daily granularity (time part is always 00:00:00)

## Environment Setup

In [ ]:
import subprocess, sys
for imp, pkg in {"kagglehub":"kagglehub","statsforecast":"statsforecast",
                  "mlforecast":"mlforecast","lightgbm":"lightgbm",
                  "lazypredict":"lazypredict","flaml":"flaml[automl]"}.items():
    try: __import__(imp)
    except ImportError: subprocess.check_call([sys.executable,"-m","pip","install","-q",pkg])
print("Packages ready.")

## Imports

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import os, pathlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

from sklearn.metrics import mean_absolute_error, mean_squared_error
from lazypredict.Supervised import LazyRegressor
from flaml import AutoML
from statsforecast import StatsForecast
from statsforecast.models import AutoARIMA, AutoETS, AutoTheta, SeasonalNaive
from mlforecast import MLForecast
from lightgbm import LGBMRegressor
from window_ops.rolling import rolling_mean

pd.set_option("display.max_columns", 30)
plt.rcParams.update({"figure.figsize": (14, 5), "axes.grid": True})
sns.set_theme(style="whitegrid")

def metrics(actual, pred, label):
    a = np.asarray(actual).ravel(); p = np.asarray(pred).ravel()
    n = min(len(a), len(p)); a, p = a[:n], p[:n]
    mae  = mean_absolute_error(a, p)
    rmse = np.sqrt(mean_squared_error(a, p))
    mask = (a != 0)
    mape = np.mean(np.abs((a[mask]-p[mask])/a[mask]))*100 if mask.sum()>0 else float("nan")
    print(f"{label:<40s}  MAE={mae:>9,.2f}  RMSE={rmse:>9,.2f}  MAPE={mape:>7.2f}%")
    return {"Model":label,"MAE":round(mae,2),"RMSE":round(rmse,2),"MAPE":round(mape,2)}

print("Imports OK.")

## Configuration

In [ ]:
KAGGLE_SLUG   = "milanzdravkovic/pharma-sales-data"
DATE_COL      = "datum"
PRIMARY_DRUG  = "M01AB"     # Change to any of M01AE, N02BA, N02BE, N05B, N05C, R03, R06
ALL_DRUGS     = ["M01AB","M01AE","N02BA","N02BE","N05B","N05C","R03","R06"]
FREQ          = "D"
SEASON_LEN    = 7
HORIZON       = 14
TEST_DAYS     = 14
VAL_DAYS      = 14
FLAML_BUDGET  = 90
RANDOM_STATE  = 42
print(f"Primary drug: {PRIMARY_DRUG}")

## Kaggle Credential Check

In [ ]:
import os, pathlib as _pl
_ok = False
if os.environ.get("KAGGLE_USERNAME") or os.environ.get("KAGGLE_KEY") or os.environ.get("KAGGLE_API_TOKEN"):
    print("[OK] Kaggle env vars found."); _ok = True
_j = _pl.Path.home() / ".kaggle" / "kaggle.json"
if _j.exists():
    print(f"[OK] kaggle.json at {_j}"); _ok = True
if not _ok:
    print("WARNING: Kaggle credentials missing.")
    print("  Option A: set KAGGLE_USERNAME + KAGGLE_KEY env vars")
    print("  Option B: place kaggle.json at ~/.kaggle/kaggle.json")
    raise EnvironmentError("Set Kaggle credentials and restart.")

## Dataset Download & Loading

In [ ]:
import kagglehub
data_path=pathlib.Path(kagglehub.dataset_download(KAGGLE_SLUG))
csv_files=sorted(data_path.rglob("*.csv"))
print([f.name for f in csv_files])
df=pd.read_csv(csv_files[0])
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
df.head()

## Data Validation

In [ ]:
# Parse datum
df[DATE_COL]=pd.to_datetime(df[DATE_COL],errors="coerce",infer_datetime_format=True)
print(f"Date range: {df[DATE_COL].min().date()} → {df[DATE_COL].max().date()}")
print(f"Parse failures: {df[DATE_COL].isna().sum()}")

# Detect which drug columns exist
available_drugs=[c for c in ALL_DRUGS if c in df.columns]
print(f"\nAvailable drug columns ({len(available_drugs)}): {available_drugs}")
if PRIMARY_DRUG not in available_drugs:
    PRIMARY_DRUG=available_drugs[0] if available_drugs else df.select_dtypes(include=[np.number]).columns[0]
    print(f"Primary drug reset to: {PRIMARY_DRUG}")

print(f"\n{PRIMARY_DRUG} stats:")
print(df[PRIMARY_DRUG].describe().round(4))
print(f"\nMissing values: {df[PRIMARY_DRUG].isna().sum()}")

In [ ]:
# Sort by date and build clean series
df=df.dropna(subset=[DATE_COL]).sort_values(DATE_COL).reset_index(drop=True)
daily_raw=df[[DATE_COL,PRIMARY_DRUG]].rename(columns={DATE_COL:"ds",PRIMARY_DRUG:"y"})
daily_raw=daily_raw.sort_values("ds").reset_index(drop=True)
full_idx=pd.date_range(daily_raw["ds"].min(),daily_raw["ds"].max(),freq="D")
daily=daily_raw.set_index("ds").reindex(full_idx).reset_index()
daily.columns=["ds","y"]
daily["y"]=daily["y"].interpolate("linear").clip(lower=0)
print(f"Series: {len(daily)} days")
print(daily["y"].describe().round(4))

## EDA — All Drug Categories

In [ ]:
# Full series for primary drug
fig=px.line(daily,x="ds",y="y",
            title=f"Daily Sales — {PRIMARY_DRUG} (NSAIDs)",
            labels={"ds":"Date","y":"Units Sold"})
fig.update_layout(template="plotly_white"); fig.show()

In [ ]:
# Compare all available drugs (normalised)
if len(available_drugs) > 1:
    df_all=df[[DATE_COL]+available_drugs].rename(columns={DATE_COL:"ds"})
    df_norm=df_all.set_index("ds")
    # Normalise each drug to 0-1 for comparison
    df_norm=(df_norm-df_norm.min())/(df_norm.max()-df_norm.min())
    df_norm=df_norm.reset_index().melt(id_vars="ds",var_name="Drug",value_name="sales_norm")
    fig=px.line(df_norm,x="ds",y="sales_norm",color="Drug",
                title="Normalised Daily Sales — All Drug Categories",
                labels={"ds":"Date","sales_norm":"Normalised Sales"})
    fig.update_layout(template="plotly_white",height=450); fig.show()
    print("Note: R06 (antihistamines) should show spring peak; N02BE (paracetamol) winter peak")

In [ ]:
# Seasonal patterns by month
daily["month"]=daily["ds"].dt.month
monthly_avg=daily.groupby("month")["y"].mean()
fig=px.bar(x=monthly_avg.index,y=monthly_avg.values,
           title=f"Average Daily Sales by Month — {PRIMARY_DRUG}",
           labels={"x":"Month","y":"Avg Units/Day"})
fig.update_layout(template="plotly_white"); fig.show()

# Day of week
daily["dow"]=daily["ds"].dt.day_name()
dow_order=["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]
fig2=px.box(daily[daily["y"]>0],x="dow",y="y",category_orders={"dow":dow_order},
            title=f"Sales Distribution by Day of Week — {PRIMARY_DRUG}")
fig2.update_layout(template="plotly_white"); fig2.show()

In [ ]:
if len(daily)>3*SEASON_LEN:
    ts_d=daily.set_index("ds")["y"].asfreq("D").interpolate()
    decomp=seasonal_decompose(ts_d,model="additive",period=SEASON_LEN)
    fig,axes=plt.subplots(4,1,figsize=(14,10),sharex=True)
    decomp.observed.plot(ax=axes[0],title="Observed"); decomp.trend.plot(ax=axes[1],title="Trend")
    decomp.seasonal.plot(ax=axes[2],title="Seasonal (weekly)"); decomp.resid.plot(ax=axes[3],title="Residual")
    plt.suptitle(f"Seasonal Decomposition — {PRIMARY_DRUG}",y=1.01)
    plt.tight_layout(); plt.show()

In [ ]:
adf=adfuller(daily["y"].dropna())
print(f"ADF p={adf[1]:.4f} → {'Stationary' if adf[1]<0.05 else 'Non-stationary'}")
fig,axes=plt.subplots(1,2,figsize=(14,4))
plot_acf(daily["y"],lags=min(60,len(daily)//3),ax=axes[0])
plot_pacf(daily["y"],lags=min(60,len(daily)//3),ax=axes[1])
plt.tight_layout(); plt.show()

## Target Analysis

In [ ]:
y=daily["y"]
print(f"Mean={y.mean():.4f}  Std={y.std():.4f}  CV={y.std()/y.mean():.3f}")
print(f"Zero days={(y==0).sum()}")
fig,ax=plt.subplots(1,2,figsize=(14,4))
ax[0].hist(y,bins=40,color="steelblue",edgecolor="white"); ax[0].set_title("Sales Distribution")
pd.plotting.lag_plot(y,lag=7,ax=ax[1],alpha=0.3); ax[1].set_title("Lag-7 Plot")
plt.tight_layout(); plt.show()

## Train / Validation / Test Split

In [ ]:
n=len(daily); test_start=n-TEST_DAYS; val_start=test_start-VAL_DAYS
ts_train=daily.iloc[:val_start].copy(); ts_val=daily.iloc[val_start:test_start].copy()
ts_test=daily.iloc[test_start:].copy(); ts_trainval=daily.iloc[:test_start].copy()
print(f"Train={len(ts_train)} | Val={len(ts_val)} | Test={len(ts_test)}")
assert ts_train["ds"].max()<ts_val["ds"].min()
assert ts_val["ds"].max()<ts_test["ds"].min()

## Preprocessing

In [ ]:
# Values in this dataset are typically units/packages sold in the 0–10 range per hour
# The daily series may be in decimal (aggregated differently) — verify
print(f"Min: {daily['y'].min():.4f}  Max: {daily['y'].max():.4f}")
print(f"Are values in daily units or normalized? Check against dataset documentation.")
print(f"Zero days: {(daily['y']==0).sum()}")
# If there are fractional values <1, the original data is hourly; sum to daily is appropriate
if daily["y"].median() < 1:
    print("Warning: Values appear to be sub-unit scale. Check if this is hourly data summed to daily.")

## Feature Engineering

In [ ]:
def build_feats(df_in):
    df=df_in.copy().sort_values("ds").reset_index(drop=True); y=df["y"]
    for lag in [1,7,14,21,28]: df[f"lag_{lag}"]=y.shift(lag)
    for w in [7,14,28]:
        df[f"rmean_{w}"]=y.shift(1).rolling(w).mean()
        df[f"rstd_{w}"]=y.shift(1).rolling(w).std()
    df["dow"]=df["ds"].dt.dayofweek
    df["is_weekend"]=(df["dow"]>=5).astype(int)
    df["month"]=df["ds"].dt.month; df["quarter"]=df["ds"].dt.quarter
    # Drug-specific: winter indicator (Nov-Feb for respiratory)
    df["is_winter"]=df["month"].isin([11,12,1,2]).astype(int)
    # Spring indicator for antihistamines
    df["is_spring"]=df["month"].isin([3,4,5]).astype(int)
    return df
full_feat=build_feats(daily)
feat_train=full_feat.iloc[:val_start].dropna().copy()
feat_val=full_feat.iloc[val_start:test_start].dropna().copy()
feat_test=full_feat.iloc[test_start:].dropna().copy()
feat_trainval=full_feat.iloc[:test_start].dropna().copy()
FEAT_COLS=[c for c in feat_train.columns if c not in ["ds","y"]]
print(f"Features ({len(FEAT_COLS)}): {FEAT_COLS}")

## Baselines, LazyPredict, FLAML

In [ ]:
results=[]; y_test=ts_test["y"].values; last_tv=ts_trainval["y"].values
results.append(metrics(y_test,np.full(TEST_DAYS,last_tv[-1]),"Naive"))
sn=np.tile(last_tv[-SEASON_LEN:],TEST_DAYS//SEASON_LEN+1)[:TEST_DAYS]
results.append(metrics(y_test,sn,"Seasonal Naive (7-day)"))
results.append(metrics(y_test,np.full(TEST_DAYS,last_tv[-7:].mean()),"MA-7"))

try:
    lz=LazyRegressor(verbose=0,ignore_warnings=True)
    lm,_=lz.fit(feat_train[FEAT_COLS],feat_val[FEAT_COLS],feat_train["y"],feat_val["y"])
    print(lm.sort_values("RMSE").head(10).to_string())
except Exception as e: print(f"LazyPredict: {e}")

flaml=AutoML()
flaml.fit(feat_trainval[FEAT_COLS],feat_trainval["y"],task="regression",metric="rmse",
          time_budget=FLAML_BUDGET,verbose=0,seed=RANDOM_STATE)
print(f"Best FLAML: {flaml.best_estimator}")
flaml_pred=flaml.predict(feat_test[FEAT_COLS]) if len(feat_test)>0 else None
if flaml_pred is not None:
    results.append(metrics(feat_test["y"].values,flaml_pred,f"FLAML ({flaml.best_estimator})"))

## StatsForecast — AutoARIMA, AutoETS, AutoTheta

In [ ]:
sf_df=ts_trainval[["ds","y"]].assign(unique_id=PRIMARY_DRUG)
sf=StatsForecast(models=[AutoARIMA(season_length=SEASON_LEN),
                          AutoETS(season_length=SEASON_LEN),
                          AutoTheta(season_length=SEASON_LEN),
                          SeasonalNaive(season_length=SEASON_LEN)],
                 freq=FREQ,n_jobs=1)
sf.fit(sf_df); sf_fc=sf.forecast(h=HORIZON)
for col in ["AutoARIMA","AutoETS","AutoTheta","SeasonalNaive"]:
    if col in sf_fc.columns:
        results.append(metrics(y_test,sf_fc[col].values[:TEST_DAYS],f"SF-{col}"))

## MLForecast

In [ ]:
from mlforecast import MLForecast
from lightgbm import LGBMRegressor
from window_ops.rolling import rolling_mean
mlf_df=ts_trainval[["ds","y"]].assign(unique_id=PRIMARY_DRUG)
mlf=MLForecast(models={"lgbm":LGBMRegressor(n_estimators=300,learning_rate=0.05,
                                              verbosity=-1,random_state=RANDOM_STATE)},
               freq=FREQ,lags=[1,7,14,28],lag_transforms={1:[(rolling_mean,7),(rolling_mean,28)]},
               date_features=["dayofweek","month"],num_threads=2)
mlf.fit(mlf_df); mlf_fc=mlf.predict(h=HORIZON)
results.append(metrics(y_test,mlf_fc["lgbm"].values[:TEST_DAYS],"MLF-LightGBM"))

## Top 3 & Forecast

In [ ]:
res_df=pd.DataFrame(results).sort_values("RMSE").reset_index(drop=True)
print(res_df.to_string()); top3=res_df.head(3)
print("\n>>> TOP 3 <<<"); print(top3.to_string(index=False))
fig=px.bar(res_df,x="Model",y="RMSE",color="RMSE",color_continuous_scale="RdYlGn_r",
           title=f"Pharmacy Demand Forecast ({PRIMARY_DRUG}) — Model Comparison")
fig.update_layout(template="plotly_white",xaxis_tickangle=-35); fig.show()

fig2=go.Figure()
fig2.add_trace(go.Scatter(x=ts_trainval["ds"].tail(60),y=ts_trainval["y"].tail(60),
    name="Train",line=dict(color="royalblue",dash="dot",width=1)))
fig2.add_trace(go.Scatter(x=ts_test["ds"],y=ts_test["y"],name="Actual",line=dict(color="black",width=2)))
preds={}
if flaml_pred is not None: preds[f"FLAML ({flaml.best_estimator})"]=flaml_pred
for col in ["AutoARIMA","AutoETS","AutoTheta"]:
    if col in sf_fc.columns: preds[f"SF-{col}"]=sf_fc[col].values[:TEST_DAYS]
if "lgbm" in mlf_fc.columns: preds["MLF-LightGBM"]=mlf_fc["lgbm"].values[:TEST_DAYS]
preds["Seasonal Naive (7-day)"]=sn
colors=["#e15759","#f28e2b","#59a14f"]
for i,(_,row) in enumerate(top3.iterrows()):
    m=row["Model"]
    if m in preds:
        fig2.add_trace(go.Scatter(x=ts_test["ds"][:len(preds[m])],y=preds[m],
            name=f"#{i+1} {m}",line=dict(color=colors[i],width=2)))
fig2.update_layout(title=f"Top 3 Pharmacy Forecasts vs Actual — {PRIMARY_DRUG}",
                   template="plotly_white"); fig2.show()

## Error Analysis

In [ ]:
best_m=top3.iloc[0]["Model"]
if best_m in preds:
    bp=np.asarray(preds[best_m]).ravel(); ba=y_test[:len(bp)]; err=ba-bp
    print(f"Bias={err.mean():+.4f}  Std={err.std():.4f}")
    fig,ax=plt.subplots(1,2,figsize=(14,4))
    ax[0].hist(err,bins=15,color="steelblue",edgecolor="white")
    ax[0].axvline(0,color="red",linestyle="--"); ax[0].set_title("Error Distribution")
    ax[1].scatter(ba,bp,s=50,alpha=0.7,color="steelblue")
    lim=max(ba.max(),bp.max())*1.1 if ba.max()>0 else 1
    ax[1].plot([0,lim],[0,lim],"r--")
    ax[1].set_xlabel("Actual"); ax[1].set_ylabel("Predicted"); ax[1].set_title("Actual vs Predicted")
    plt.tight_layout(); plt.show()

## Insights & Interpretation
1. **`M01AB` (NSAIDs)** shows mild winter peak and stable weekly pattern — relatively easy to forecast
2. **Weekly seasonality** from pharmacy opening hours (closed Sundays in many countries)
3. **Seasonal drugs** (`R06` antihistamines, `R03` bronchodilators) are harder — require annual pattern data
4. **AutoTheta** and **AutoETS** often win on pharmaceutical daily series due to their
   exponential smoothing component, which handles the mild trend gracefully
5. **Stock-out asymmetry**: running out of NSAIDs is worse than over-stocking (patient harm risk)

## Limitations
1. Single-pharmacy data — regional patterns may differ
2. No prescriptions vs OTC split — prescription drugs have different demand drivers
3. No competitor/substitution data — when one NSAID is unavailable, another is substituted
4. Annual seasonality needs 2+ years of data to estimate reliably with daily resolution

## How to Improve
1. Forecast all 8 drug categories simultaneously using MLForecast panel mode
2. Add a prescription-volume feature from pharmacy management system
3. Use `CrostonOptimised` from StatsForecast for intermittent/low-volume drugs
4. Incorporate temperature as a covariate for respiratory drugs (N02BE, R03)
5. Build safety-stock alert: forecast + 1.5σ as reorder trigger

## Production Considerations
1. **Multi-series forecasting**: In production, forecast all 8 drug categories simultaneously using MLForecast panel mode
2. **Safety stock buffer**: Use quantile forecasts (e.g. 90th percentile) as the reorder trigger, not point forecasts
3. **Expiry-aware ordering**: Combine demand forecast with shelf-life to avoid over-ordering perishable drugs
4. **Distributor lead time**: Forecast horizon must exceed lead time from wholesaler (typically 1–3 days)
5. **Regulatory compliance**: Controlled substances (N05B/N05C) have different ordering rules — forecasts must respect legal constraints
6. **Model monitoring**: Track MAE weekly; retrain monthly or when seasonal patterns shift (e.g. early flu season)

## Common Mistakes
1. Treating the `datum` time component as meaningful (it's always 00:00:00)
2. Using raw daily values without checking if they represent per-unit or per-package
3. Applying the same model to seasonal drugs (R06) and flat drugs (N05B) — different models needed
4. Ignoring the pharmacy closure schedule (Sunday zeros are structural, not noise)

## Exercises
1. Forecast `R06` (antihistamines) and compare seasonality with `M01AB`
2. Add month-of-year as a one-hot feature and check if it improves annual-seasonal drugs
3. Compute safety stock = forecast + 2σ; plot alongside actual to see buffer adequacy
4. Use MLForecast to forecast all 8 drug categories simultaneously in one pass

## Summary
- Worked with real pharmaceutical sales data (ATC codes, 8 drug categories)
- Focused on `M01AB` (NSAIDs) as the primary target
- EDA revealed drug-specific seasonality, day-of-week pharmacy closure effects
- Compared 5 forecasting approaches; top-3 selected by test RMSE
- Key lesson: pharmaceutical demand is relatively stable — but seasonal patterns
  differ dramatically across drug categories (winter vs allergy vs flat)

---
*Notebook #15 of 50 — Time Series Forecasting Portfolio*